# 08 — Cross-modal consensus analysis

Reads the JSONL audit at `logs/visual_consensus.jsonl` produced by `cross_modal_validator.CrossModalValidator` and reports:
1. Modality coverage (how often each modality is present)
2. Pairwise Cohen's Kappa (satellite vs field, satellite vs features, field vs features)
3. Fleiss' Kappa over the 3-way agreement matrix
4. Disagreement patterns (which class-pairs disagree most)
5. Flag distribution (HIGH_CONF_OK / INVESTIGATE / EARLY_WARNING / ...)
6. Daily NDVI overlay + consensus class strip plot.

In [ ]:
import os, sys, json
import pandas as pd
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from visual_validation import config
from visual_validation.consensus import agreement_metrics as am
from pathlib import Path

rows = []
for line in Path(config.CONSENSUS_LOG).read_text(encoding='utf-8').splitlines():
    try:
        rows.append(json.loads(line))
    except Exception:
        pass
df = pd.DataFrame(rows)
print('rows:', len(df))
df.head()

In [ ]:
# Pull per-modality class out of the nested modalities dict
for m in ['satellite', 'field', 'features']:
    df[m] = df['modalities'].apply(lambda d: (d.get(m) or {}).get('class') if isinstance(d, dict) else None)
df[['site_id', 'target_date', 'satellite', 'field', 'features', 'consensus_class', 'flag']].head()

In [ ]:
records = df[['satellite', 'field', 'features']].to_dict('records')
summary = am.summarise(records)
print('n_total =', summary.n_records_total)
print('n_all_three =', summary.n_records_all_three)
print('fleiss_kappa =', summary.fleiss_kappa, '->', summary.fleiss_interpretation)
pd.DataFrame(summary.pairwise_kappa)

In [ ]:
pd.DataFrame(summary.disagreement_top)

In [ ]:
df['flag'].value_counts()

In [ ]:
# Strip plot: consensus class over time per site
import plotly.express as px
df['target_date'] = pd.to_datetime(df['target_date'])
fig = px.scatter(df, x='target_date', y='consensus_class', color='flag',
                 facet_col='site_id', title='Consensus class over time')
fig.show()